In [ ]:
#%wget https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/TP_PS/m2_ml2_tp_perceptron_structure.zip
#%unzip m2_ml2_tp_perceptron_structure.zip

In [32]:
import sys
sys.path.append('/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_APPAUT/TP2')

In [36]:
DATA_PATH = "/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_APPAUT/TP2/data"

import pos_corpus as pcc
import id_feature as idfc

import discriminative_sequence_classifier as dsc
import numpy as np

Sujet TP2

# Perceptron structuré pour du POS tagging

Le POS tagging est une tâche de classification dite structurée. Il s'agit de prédire les tags syntaxiques (POS tags) des mots d'une phrase.  

# Chargement du dataset CoNLL

In [37]:
corpus = pcc.PostagCorpus()
train_seq = corpus.read_sequence_list_conll(DATA_PATH + "/train-02-21.conll",
                                            max_sent_len=10, max_nr_sent=1000)

test_seq = corpus.read_sequence_list_conll(DATA_PATH + "/test-23.conll",
                                           max_sent_len=10, max_nr_sent=1000)

# nous n'utlisons pas le dev ici
# dev_seq = corpus.read_sequence_list_conll(DATA_PATH + "/dev-22.conll",
#                                           max_sent_len=10, max_nr_sent=1000)


Afficher des exemples de séquences.

In [38]:
# on regarde des exemples de phrases
seq_ind = 3
seq = train_seq[seq_ind]

print(seq)
print('x', seq.x)
print('y', seq.y)
print('x[0]', seq.x[0])
print('y[0]', seq.y[0])


Not/adv all/det those/det who/pron wrote/verb oppose/verb the/det changes/noun ./. 
x [288, 289, 290, 259, 219, 291, 19, 162, 41]
y [8, 2, 2, 9, 6, 6, 2, 0, 4]
x[0] 288
y[0] 8


# Les *feature functions* ou *potentials* 

Nous allons dans un premier temps utiliser des feature functions qui mimiquent les HMM : 

**Initial features** : 
 - features en position 0 dans les phrases qui encodent l'identité des tags en position 0, exemple : init_tag:pron 

**Emission features** : 
 - features mot/tag, appelés "nodes" (émission), exemple : id:many::adj 

**Transition features** : 
 - features tag/tag, appelés "edges" (transition), exemple : prev_tag:adv::num  

**Final features** : 
 - features en position finale dans les phrases: l'identité des tags en position finale, exemple : final_prev_tag:num 


Dans le fichier ```id_feature.py```, est définie une classe ```IDFeatures``` qui crée ces features à partir des séquences mot/tag du train : 

In [39]:
## Instancier et créer les feature functions sur le train 
feature_mapper = idfc.IDFeatures(train_seq)
feature_mapper.build_features()
print(f'len(feature_mapper.feature_dict) : {len(feature_mapper.feature_dict)}')

len(feature_mapper.feature_dict) : 2683


Afficher tous les noms de feature functions et les compter.

In [40]:
nb = 0
current_feature_list = feature_mapper.feature_list[seq_ind]
for el in current_feature_list:
    print(f'{type(el)} - {len(el)} - {el[:]}')
    nb += 1
print(nb)

<class 'list'> - 1 - [[37]]
<class 'list'> - 8 - [[40], [42], [44], [46], [19], [49], [28], [9]]
<class 'list'> - 1 - [[10]]
<class 'list'> - 9 - [[38], [39], [41], [43], [45], [47], [48], [50], [8]]
4


Afficher tous les noms de feature functions de la séquence d'indice ```seq_ind```.

In [41]:
current_feature_list = feature_mapper.feature_list[seq_ind]
for el in current_feature_list:
    for e in el : 
        print(feature_mapper.feature_dict.get_label_name(e[0]))

init_tag:adv
prev_tag:adv::det
prev_tag:det::det
prev_tag:det::pron
prev_tag:pron::verb
prev_tag:verb::verb
prev_tag:verb::det
prev_tag:det::noun
prev_tag:noun::.
final_prev_tag:.
id:Not::adv
id:all::det
id:those::det
id:who::pron
id:wrote::verb
id:oppose::verb
id:the::det
id:changes::noun
id:.::.


# Le perceptron structuré (Collins, 2002)

Le perceptron structuré est un classifieur de séquences à séquences, dit discriminant. 

Ce type d'approche modélise : 
 - la probabilité conditionnelle d'une séquence de tags $\boldsymbol y$ étant donnée une séquence de mots $\boldsymbol x$, 
 - cette modélisation s'effectue à l'aide de produits scalaires entre le vecteur de poids $\boldsymbol w$ (à apprendre) 
 - et les *feature functions*, dont des exemples ont été donnés ci-dessus :  

\begin{equation} 
P(\boldsymbol y | \boldsymbol x ; \boldsymbol w) = \displaystyle\frac{1}{Z(\boldsymbol w, \boldsymbol x)}\exp \Big( \boldsymbol w \cdot \boldsymbol f_{\text{init}}(\boldsymbol x, y_0)+\sum_{i=0}^{N-2}\boldsymbol w \cdot \boldsymbol f_{\text{trans}}(i, \boldsymbol x, y_i, y_{i+1}) +\boldsymbol w  \cdot \boldsymbol f_{\text{final}}(\boldsymbol x, y_{N-1}) + \sum_{i=0}^{N-1}\boldsymbol w \cdot \boldsymbol f_{\text{emission}}(i, \boldsymbol x, y_i)\Big) 
\end{equation} 



Nous vous fournissons un squelette du code du perceptron structuré dans une cellule ci-dessous, à compléter.


La classe ```StructuredPerceptron``` hérite de la classe ```DiscriminativeSequenceClassifier```, qui elle-même hérite de la classe ```SequenceClassifier```. 

Ces deux classes sont données dans les fichiers respectifs ```discriminative_sequence_classifier.py``` et ```sequence_classifier.py```. 

Prenez le temps de lire ce que contiennent ces fichiers.

Voici le diagramme UML qui résume ces dépendances.


<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/TP_PS/diagramme_UML_structured_perceptron.png"
     alt="Digramme UML"
     style="float: left; margin-right: 1px;"
     />

- Nous avons une séquence en entrée et nous devons la labiliser.
- Ensuite nous mettrons à jour les poids du perceptron à partir de la position nous détectons une erreur.
- Il est à considérer que les erreurs font partie de la séquence. 

- Comment pouvonsd déterminer la "feature" a pénaliser si identifions une une erreur?
- Deux choses changent lors de l'inférence, c'est ce qui motive l'utilisation de l'algorthme de Viterbi.

- A partir de là nous devons :
   - mettre à jour les poids fournis au cours de l'inférence 
   - lors de l'inférence de la séquence si le label est juste, il n' pays de mettre à jour les poids,
   - si le label est différent du vrai label (Viterbi) on pénalise la feature mais laquelle?  

Maintenant notre algorithme du perceptron structuré se représente un peu différemment (cf. ci-dessous).

- Maintenant le training positionne les lablels mis séquences. 
- On initialise le poids du vecteur à 0. 
- Nous éfinissons la fonction qui effectue la recherche pour tous les labels possibles.
- Elle fournit la valeur maximale du label ayant le plus haut score "argmax. 
- Si nous obtenons la bonne réponse on ne fait rien. 
- Dans le cas contraire on soutrait:
   - au vecteur feature de la vrai séquence de label 
   - le vecteur feature de la séquence de labels prédits via notre fonction.

## L'algorithme d'apprentissage

Initialiser les poids du perceptron avec le vecteur nul : $\boldsymbol w = \boldsymbol 0$

Pour $i=1\ldots T$

*   Pour chaque exemple d'apprentissage ($\boldsymbol x, \boldsymbol y$)

    1.    Générer une séquence de prédictions : $\boldsymbol z = argmax_{\boldsymbol z} \boldsymbol w \cdot 
\boldsymbol f (\boldsymbol x, \boldsymbol y)$
                 
    2.    Pour chaque Si $\boldsymbol z \neq \boldsymbol y$, faire : 
                 
\begin{equation*}
                    \boldsymbol w \leftarrow \boldsymbol w + \boldsymbol f (\boldsymbol x, \boldsymbol y) - \boldsymbol f (\boldsymbol x, \boldsymbol z)
\end{equation*}



La fonction $\boldsymbol f$ correspond aux feature functions extraits pour les séquences $\boldsymbol x, \boldsymbol z$, et sont accessibles à l'aide de l'objet ```feature_mapper```, vu ci-dessus.


## Travail à faire


Vous devez coder la méthode ```perceptron_update()``` de la classe ```StructuredPerceptron``` de la cellule ci-dessous.


Cette fonction prend en entrée ***une séquence*** et effectue les lignes 1 et 2 de l'algorithme sur cette séquence. 

Elle retourne ```num_labels, num_mistakes``` qui sont respectivement le nombre d'éléments de la séquence à traiter et le nombre d'erreurs commises par le modèle sur la séquence.

Détaillons ces deux lignes : 

1.   Pour générer la séquence de prédictions $\boldsymbol z$, faire un décodage Viterbi sur la séquence.

2.   Le vecteur de poids $\boldsymbol w$ du perceptron correspond à ```self.parameters```.

La mise à jour du vecteur est faite en testant chaque élément de la séquence prédite $z_i$ avec $i=0\ldots L-1$, avec $L$ la longueur de la séquence.     
Les features étant tous binaires, si une prédiction est fausse pour une position $i$, alors il faut ajouter ou retrancher 1 aux quatre types de feature functions.   

Plus précisément, pour la première position dans la séquence ($i=0$) :
 - si $z_0 \neq y_0$, faire : 

   - $w[\text{initial features}(( x, y_0))] \mathrel{+}= 1$
   - $w[\text{initial features}(( x, z_0))] \mathrel{-}= 1$
    
 - Puis pour $i=0\ldots L-1$ : 
    - si $z_i \neq y_i$, modifier $ \boldsymbol w$ de la même façon mais en considérant les *emission features*  et les *transition features*.
    - Attention, les *transition features* ne sont pertinents que pour $i>0$.
 - Enfin pour la dernière position $i=L-1$, il faut aussi considérer les *final features*.


***Aide***

Pour récupérer les quatre types de feature functions, ```feature_mapper``` a les méthodes suivantes :  

*    ```get_initial_features(sequence, y)``` 
*    ```get_emission_features(sequence, i, y)``` 
*    ```get_transition_features(sequence, i, y, y_prev)``` 
*    ```get_final_features(sequence, y_prev)``` 

Avec $i$ la position dans une séquence, ```y``` le tag à la position ```i``` de la vérité terrian ou bien issu de la prédiction, et ```y_prev``` le tag à la position ```i-1```, lorsqu'elle existe.

\begin{equation}
P(O^0, \ldots, O^{T-1}, S^0, \ldots, S^{T-1}) = \boldsymbol \pi(S^0) \, P(O^0|S^0)\,Π_1^{T-1}\,\,P(S^{t}|S^{t-1})\,P(O^t|S^T)
\end{equation}

In [42]:
class StructuredPerceptron(dsc.DiscriminativeSequenceClassifier):
    """ Implements Structured Perceptron"""

    def __init__(self, observation_labels, state_labels, feature_mapper,
                 num_epochs=10, learning_rate=1.0, averaged=True):
      
        dsc.DiscriminativeSequenceClassifier.__init__(self, observation_labels, state_labels, feature_mapper)
        self.num_epochs = num_epochs
        self.learning_rate = learning_rate
        self.averaged = averaged
        self.params_per_epoch = []

    def train_supervised(self, dataset):
        self.parameters = np.zeros(self.feature_mapper.get_num_features())
        num_examples = dataset.size()
        for epoch in range(self.num_epochs):
            num_labels_total = 0
            num_mistakes_total = 0
            for i in range(num_examples):
                sequence = dataset.seq_list[i]
                num_labels, num_mistakes = self.perceptron_update(sequence)
                num_labels_total += num_labels
                num_mistakes_total += num_mistakes
            self.params_per_epoch.append(self.parameters.copy())
            acc = 1.0 - num_mistakes_total / num_labels_total
            print("Epoch: %i Accuracy: %f" % (epoch, acc))
        self.trained = True

        if self.averaged:
            new_w = 0
            for old_w in self.params_per_epoch:
                new_w += old_w
            new_w /= len(self.params_per_epoch)
            self.parameters = new_w

    def perceptron_update(self, sequence):
        # Exercice Z issue de viterbi decode
        num_labels, num_mistakes = 0, 0
        
        w = self.parameters
        z_pred, total_score = self.viterbi_decode(sequence)
        num_labels = len(z_pred)-1
      # z_pred.add_emission_features
      # self.feature_mapper
      # get_transition_features(sequence, i, y, y_prev)

        z_pred_yi = self.feature_mapper.get_initial_features(sequence, z_pred.y[0])
        s_sequ_yi = self.feature_mapper.get_initial_features(sequence, sequence.y[0]) 

        if z_pred_yi != s_sequ_yi :
            num_mistakes += 1
            w[s_sequ_yi] += 1               
            w[z_pred_yi] -= 1

        z_pred_yi = self.feature_mapper.get_emission_features(sequence, 0, z_pred.y[0])
        s_sequ_yi = self.feature_mapper.get_emission_features(sequence, 0, sequence.y[0]) 

        if z_pred_yi != s_sequ_yi :
            num_mistakes += 1
            w[s_sequ_yi] += 1               
            w[z_pred_yi] -= 1             
        
        for i in range (1,num_labels):

            z_pred_yi = self.feature_mapper.get_transition_features(sequence, i, z_pred.y[i], z_pred.y[i-1]) 
            s_sequ_yi = self.feature_mapper.get_transition_features(sequence, i, sequence.y[i], sequence.y[i-1]) 

            if z_pred_yi != s_sequ_yi :
               num_mistakes += 1
               w[s_sequ_yi] += 1               
               w[z_pred_yi] -= 1  

            z_pred_yi = self.feature_mapper.get_emission_features(sequence, i, z_pred.y[i])
            s_sequ_yi = self.feature_mapper.get_emission_features(sequence, i, sequence.y[i])            

            if z_pred_yi != s_sequ_yi :
               num_mistakes += 1
               w[s_sequ_yi] += 1               
               w[z_pred_yi] -= 1                                               

        z_pred_yi = self.feature_mapper.get_transition_features(sequence, num_labels, z_pred.y[num_labels], z_pred.y[num_labels-1]) 
        s_sequ_yi = self.feature_mapper.get_transition_features(sequence, num_labels, sequence.y[num_labels], sequence.y[num_labels-1]) 

        if z_pred_yi != s_sequ_yi :
            num_mistakes += 1
            w[s_sequ_yi] += 1               
            w[z_pred_yi] -= 1    
            
        z_pred_yi = self.feature_mapper.get_final_features(sequence, z_pred.y[num_labels])   
        s_sequ_yi = self.feature_mapper.get_final_features(sequence, sequence.y[num_labels])   

        if z_pred_yi != s_sequ_yi :
            num_mistakes += 1
            w[s_sequ_yi] += 1               
            w[z_pred_yi] -= 1

        z_pred_yi = self.feature_mapper.get_emission_features(sequence, num_labels, z_pred.y[num_labels])
        s_sequ_yi = self.feature_mapper.get_emission_features(sequence, num_labels, sequence.y[num_labels])            

        if z_pred_yi != s_sequ_yi :
            num_mistakes += 1
            w[s_sequ_yi] += 1               
            w[z_pred_yi] -= 1 

        return num_labels, num_mistakes

    def save_model(self, dir):
        fn = open(dir + "parameters.txt", 'w')
        for p_id, p in enumerate(self.parameters):
            fn.write("%i\t%f\n" % (p_id, p))
        fn.close()

    def load_model(self, dir):
        fn = open(dir + "parameters.txt", 'r')
        for line in fn:
            toks = line.strip().split("\t")
            p_id = int(toks[0])
            p = float(toks[1])
            self.parameters[p_id] = p
        fn.close()


 - Instancier le perceptron.
 - Réaliser un entraînement sur 10 epochs : 

In [43]:
sp = StructuredPerceptron(corpus.word_dict, corpus.tag_dict,feature_mapper)
sp.train_supervised(train_seq)

Epoch: 0 Accuracy: -0.045902
Epoch: 1 Accuracy: 0.418119
Epoch: 2 Accuracy: 0.594996
Epoch: 3 Accuracy: 0.689215
Epoch: 4 Accuracy: 0.747368
Epoch: 5 Accuracy: 0.796721
Epoch: 6 Accuracy: 0.816566
Epoch: 7 Accuracy: 0.840725
Epoch: 8 Accuracy: 0.858326
Epoch: 9 Accuracy: 0.846592


Réaliser un décodage Viterbi et une évaluation sur le corpus de test. 
- Afficher l'accuracy.

In [44]:
viterbi_pred  = sp.viterbi_decode_corpus(test_seq)
acc = sp.evaluate_corpus(test_seq, viterbi_pred)
print("Décodage Viterbi subset Test, acc=%.2f%%"%(100.*acc))

Décodage Viterbi subset Test, acc=84.17%


Réaliser un décodage Posterior et une évaluation sur le corpus de test.  
 - Afficher l'accuracy. 
 - Est-ce meilleur que Viterbi ?

In [45]:
post_pred = sp.posterior_decode_corpus(test_seq)
sp.evaluate_corpus(test_seq, post_pred)
print("Décodage Posterior subset Test, acc=%.2f%%"%(100.*acc))

Décodage Posterior subset Test, acc=84.17%


Y-a-t-il du sur-apprentissage ? 

**A priori oui le score est meilleur**

# Jeu de *feature functions* étendu


Tester à nouveau le perceptron mais cette fois avec les feature functions étendues mises à disposition dans le fichier ```extended_feature.py```.


Quels types de feature functions sont ajoutées par rapport à la classe précédente ```IDFeatures``` ?



Je n'ai pas remarqué qu'il y avait de nouvelle méthode seul la méthode add_emission_features a été surchargée prenants en compte la possibilité de dfinitions de features plus élaborées. 

In [46]:
import extended_feature as extfc

In [47]:
## Instancier et créer les extended feature mapper functions
ext_feature_mapper = extfc.ExtendedFeatures(train_seq)
ext_feature_mapper.build_features()
print(f'len(feature_mapper.feature_dict) : {len(feature_mapper.feature_dict)}')

len(feature_mapper.feature_dict) : 2683


In [48]:
nb = 0
ext_current_feature_list = ext_feature_mapper.feature_list[seq_ind]
for el in ext_current_feature_list:
    print(f'{type(el)} - {len(el)} - {el[:]}')
    nb += 1
print(nb)

<class 'list'> - 1 - [[121]]
<class 'list'> - 8 - [[132], [139], [145], [151], [65], [159], [93], [32]]
<class 'list'> - 1 - [[33]]
<class 'list'> - 9 - [[122, 123, 103, 124, 125, 126], [127, 128, 129, 130, 131], [133, 37, 134, 135, 136, 137, 138], [140, 141, 142, 143, 144], [146, 63, 147, 148, 59, 149, 150], [152, 63, 153, 154, 155, 156, 157], [158, 37, 38, 136, 137], [160, 161, 162, 163, 164, 165, 166], [31]]
4


<class 'list'> - 1 - [[37]]
<class 'list'> - 8 - [[40], [42], [44], [46], [19], [49], [28], [9]]
<class 'list'> - 1 - [[10]]
<class 'list'> - 9 - [[38], [39], [41], [43], [45], [47], [48], [50], [8]]

In [49]:
ext_current_feature_list = ext_feature_mapper.feature_list[seq_ind]
ext_current_feature_list
for el in ext_current_feature_list:
    for e in el : 
        print(ext_feature_mapper.feature_dict.get_label_name(e[0]))

init_tag:adv
prev_tag:adv::det
prev_tag:det::det
prev_tag:det::pron
prev_tag:pron::verb
prev_tag:verb::verb
prev_tag:verb::det
prev_tag:det::noun
prev_tag:noun::.
final_prev_tag:.
id:Not::adv
id:all::det
id:those::det
id:who::pron
id:wrote::verb
id:oppose::verb
id:the::det
id:changes::noun
id:.::.


In [50]:
sp = StructuredPerceptron(corpus.word_dict, corpus.tag_dict,ext_feature_mapper)
sp.train_supervised(train_seq)

Epoch: 0 Accuracy: 0.256083
Epoch: 1 Accuracy: 0.580328
Epoch: 2 Accuracy: 0.676790
Epoch: 3 Accuracy: 0.756169
Epoch: 4 Accuracy: 0.792925
Epoch: 5 Accuracy: 0.833305
Epoch: 6 Accuracy: 0.828818
Epoch: 7 Accuracy: 0.862640
Epoch: 8 Accuracy: 0.854530
Epoch: 9 Accuracy: 0.870751
